# 📥 Notebook 1 — Data Ingestion & Structural Cleaning
**Project:** Citizen Grievance Analysis  
**Dataset:** NYC 311 Service Requests  


### What this notebook does:
- Loads the raw CSV from `data/raw/`
- Inspects shape, columns, null values
- Selects relevant columns
- Fixes missing values
- Parses dates & engineers new features
- Saves cleaned data → `outputs/data_cleaned.csv`

---

##  Install Libraries (Run once, then skip)

In [ ]:
# Uncomment and run ONCE, then comment back
# !pip install pandas numpy matplotlib seaborn nltk wordcloud scikit-learn

##  Import Libraries & Setup Paths

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Folder Paths (relative to notebooks/ folder) ─────────────────────────────
RAW_DATA_PATH = '../data/raw/311_service_requests_sample.csv'
OUTPUT_DIR    = '../outputs/'

# Create outputs folder if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Libraries imported!')
print(f'📂 Raw data path : {RAW_DATA_PATH}')
print(f'📂 Output folder : {OUTPUT_DIR}')

## Load the Raw CSV

In [ ]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
df.head(3)

##  See All Column Names

In [ ]:
for i, col in enumerate(df.columns):
    print(f'{i+1:>3}. {col}')

##  Check Missing Values

In [ ]:
null_df = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %'    : (df.isnull().sum() / len(df) * 100).round(2)
})

# Show only columns that HAVE nulls
print(null_df[null_df['Null Count'] > 0].to_string())

##  Select Only Useful Columns

In [ ]:
# We only need these 10 columns for NLP work
NLP_COLS = [
    'Unique Key',
    'Created Date',
    'Closed Date',
    'Agency Name',
    'Complaint Type',        # ← This is our TARGET LABEL for classification
    'Descriptor',            # ← Main complaint text
    'Location Type',
    'Borough',
    'Status',
    'Resolution Description' # ← How it was resolved
]

df_nlp = df[NLP_COLS].copy()
print(f'Reduced to {df_nlp.shape[1]} columns, {df_nlp.shape[0]:,} rows')
df_nlp.head(3)

## Fix Missing Values

In [ ]:
TEXT_COLS = [
    'Descriptor', 'Resolution Description',
    'Complaint Type', 'Agency Name', 'Location Type'
]

for col in TEXT_COLS:
    df_nlp[col] = df_nlp[col].fillna('unknown')

# Drop rows where Complaint Type is missing
# (it is our label — we cannot train without it)
df_nlp = df_nlp[df_nlp['Complaint Type'] != 'unknown']

print(f'Rows after fixing nulls: {len(df_nlp):,}')
print('Remaining nulls:')
print(df_nlp.isnull().sum())

## Parse Dates & Engineer Features

In [ ]:
df_nlp['Created Date'] = pd.to_datetime(df_nlp['Created Date'], errors='coerce')
df_nlp['Closed Date']  = pd.to_datetime(df_nlp['Closed Date'],  errors='coerce')

# How many hours to resolve the complaint?
df_nlp['resolution_hours'] = (
    (df_nlp['Closed Date'] - df_nlp['Created Date'])
    .dt.total_seconds() / 3600
)

# What hour of the day was complaint filed?
df_nlp['hour_created'] = df_nlp['Created Date'].dt.hour

# What day of the week?
df_nlp['day_of_week'] = df_nlp['Created Date'].dt.day_name()

print('New columns added: resolution_hours, hour_created, day_of_week')
df_nlp[['Created Date','Closed Date','resolution_hours','hour_created','day_of_week']].head()

##  Create Combined Text Column

In [ ]:
# Merge complaint type + descriptor + resolution into one text field
# This gives NLP models more signal to work with
df_nlp['combined_text'] = (
    df_nlp['Complaint Type'].astype(str)         + ' ' +
    df_nlp['Descriptor'].astype(str)             + ' ' +
    df_nlp['Resolution Description'].astype(str)
)

print('Sample combined_text:')
print(df_nlp['combined_text'].iloc[0])

##  Quick Summary & Save

In [ ]:
print('=== FINAL CLEANED DATASET SUMMARY ===')
print(f'Total rows    : {len(df_nlp):,}')
print(f'Total columns : {df_nlp.shape[1]}')
print(f'Unique complaint types (labels): {df_nlp["Complaint Type"].nunique()}')
print(f'Boroughs      : {df_nlp["Borough"].unique().tolist()}')
print(f'Date range    : {df_nlp["Created Date"].min()} → {df_nlp["Created Date"].max()}')

# Save output for Notebook 2
SAVE_PATH = OUTPUT_DIR + 'data_cleaned.csv'
df_nlp.to_csv(SAVE_PATH, index=False)
print(f'\n✅ Saved → {SAVE_PATH}')
print('👉 Now open Notebook 2: 02_text_preprocessing.ipynb')